# Lecture 14.1 Semantic Segmentation

**Topic:** semantic segmentation from intuition to real training pipelines  
**Audience:** students who know CNN basics but are new to dense prediction  
**Goal:** understand what segmentation predicts, how the main architectures work, how to evaluate them, and how to run a real example on a real dataset

This version is intentionally much more detailed than the earlier draft. The point is not only to give students runnable code, but also to make the theory readable for someone who has never worked with dense prediction before.


## Outline

1. What segmentation is and why it is different from classification  
2. Semantic vs instance vs panoptic segmentation  
3. Why segmentation is fundamentally harder than image classification  
4. The core architecture ideas: FCN, U-Net, DeepLabV3  
5. Losses and metrics students actually need  
6. Real dataset example with Oxford-IIIT Pet  
7. Practical PyTorch pipeline  
8. Applications, failure modes, and exercises


In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path

import numpy as np

MPL_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
except Exception as exc:
    MPL_AVAILABLE = False
    print(f"matplotlib is not available: {exc}")

PIL_AVAILABLE = True
try:
    from PIL import Image
except Exception as exc:
    PIL_AVAILABLE = False
    print(f"Pillow is not available: {exc}")

TORCH_AVAILABLE = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
except Exception as exc:
    TORCH_AVAILABLE = False
    print(f"PyTorch is not available: {exc}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)

print(f"matplotlib available: {MPL_AVAILABLE}")
print(f"Pillow available: {PIL_AVAILABLE}")
print(f"PyTorch available: {TORCH_AVAILABLE}")


## 1. What is semantic segmentation?

Semantic segmentation means assigning a class label to **every pixel** in the image.

That wording sounds simple, but it changes the entire learning problem. In image classification, the model produces one label for the whole image. In detection, the model predicts a set of boxes. In segmentation, the output is not a single scalar or a short list: it is a dense map whose spatial structure should line up with the original image.

This means a segmentation model must solve two tasks at once:

- understand **what** is present in the scene  
- preserve enough geometry to decide **where** the object begins and ends

This is why segmentation sits at an important intersection between recognition and localization. It is about semantics and geometry at the same time.

For beginners, a helpful mental model is:

- classification says “there is a cat somewhere”  
- detection says “the cat is approximately inside this rectangle”  
- segmentation says “these exact pixels belong to the cat”


![Segmentation pipeline](images/segmentation_pipeline.svg)

The picture above summarizes the task: an RGB image goes in, and a class mask of the same spatial meaning comes out. Even when the output is downsampled internally, the model is still trained to predict a dense per-pixel labeling.


## 2. Semantic, instance, and panoptic segmentation

Students often hear several related terms and mix them up. It is worth separating them carefully.

### Semantic segmentation

Every pixel gets a class, but different objects of the same class are not separated. If there are three cars, all car pixels share the same label.

### Instance segmentation

Now the model must separate objects individually. Car #1 and car #2 need different masks, even though they belong to the same class.

### Panoptic segmentation

This combines both worlds. The model predicts:

- “stuff” categories such as sky, road, grass  
- “thing” instances such as person 1, person 2, bicycle 1

Why does this matter pedagogically? Because students should know that semantic segmentation is the easiest entry point: it already teaches dense prediction, but avoids the extra complexity of separating instances.


In [ ]:
# A segmentation mask is just a matrix of class ids.

toy_mask = np.array(
    [
        [0, 0, 1, 1, 1],
        [0, 2, 2, 1, 1],
        [0, 2, 2, 1, 0],
        [0, 0, 0, 0, 0],
    ],
    dtype=np.int64,
)

print("Mask shape:", toy_mask.shape)
print("Unique labels:", np.unique(toy_mask))

if MPL_AVAILABLE:
    plt.figure(figsize=(5, 3))
    plt.imshow(toy_mask, cmap="viridis")
    plt.colorbar()
    plt.title("A toy semantic segmentation mask")
    plt.show()


## 3. Why segmentation is harder than classification

A classifier can aggressively downsample the image and still perform well, because it only needs a global answer. Segmentation cannot do that carelessly. The moment we compress the image too much, thin boundaries, small objects, and fine structures start to disappear.

This creates a central tension in segmentation:

- deep layers provide semantic meaning  
- shallow layers preserve spatial detail

Any successful segmentation architecture must reconcile these two needs.

From a systems perspective, this means segmentation models are designed around one key question:

**How do we keep enough context and enough resolution at the same time?**

That question explains why architectures like FCN, U-Net, and DeepLab are shaped the way they are.


## 4. A brief architecture roadmap

Before going into U-Net and DeepLabV3, it helps to see where they fit historically.

### FCN (Fully Convolutional Network)

FCN was one of the first major steps toward modern segmentation. It removed fully connected layers and turned classification-style CNNs into dense predictors. This was historically important because it showed that CNN features could be projected back to spatial outputs.

### U-Net

U-Net became hugely influential because it solved a practical problem very elegantly:

- an encoder compresses information  
- a decoder upsamples it again  
- skip connections bring back fine spatial details

This makes U-Net especially intuitive for teaching and especially strong on tasks where boundaries matter.

### DeepLab family

DeepLab focuses more on expanding the receptive field and capturing multi-scale context using dilated convolutions and ASPP. It is a strong modern baseline, especially when students are ready to connect segmentation with mainstream backbone design.


![U-Net architecture](images/unet_architecture.svg)

U-Net is worth studying carefully because it teaches almost every idea students need later:

- encoder-decoder structure  
- bottleneck reasoning  
- skip connections  
- dense prediction head  
- the tension between semantic abstraction and local detail


## 5. U-Net explained slowly

U-Net has two symmetrical halves:

### Encoder

The encoder progressively reduces the spatial resolution while increasing the number of feature channels. That trades detail for semantic abstraction. Early layers respond to edges and textures, while deeper layers respond more to object identity and context.

### Bottleneck

The bottleneck sees the image at the lowest spatial resolution. At this stage the representation contains broad contextual information, but exact boundaries have usually become blurry.

### Decoder

The decoder upsamples the feature maps back toward the original resolution. By itself, this would often produce coarse masks.

### Skip connections

This is the crucial idea. U-Net copies feature maps from the encoder into corresponding decoder stages. These skip connections give the decoder access to the high-resolution information that would otherwise be lost.

Why do students love this architecture once they understand it? Because the motivation is intuitive:

- the encoder answers “what is this?”  
- the decoder answers “where exactly is it?”  
- the skip connections keep the “where” answer from becoming too fuzzy


In [ ]:
def binary_iou(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    inter = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return float(inter / union) if union > 0 else 1.0


def binary_dice(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    inter = np.logical_and(y_true, y_pred).sum()
    denom = y_true.sum() + y_pred.sum()
    return float((2 * inter) / denom) if denom > 0 else 1.0


y_true = np.array([[0, 1, 1], [0, 1, 0], [0, 0, 0]])
y_pred = np.array([[0, 1, 0], [0, 1, 1], [0, 0, 0]])

print("IoU :", round(binary_iou(y_true, y_pred), 3))
print("Dice:", round(binary_dice(y_true, y_pred), 3))


## 6. DeepLabV3 explained in much greater detail

DeepLabV3 solves the segmentation problem from a very different angle than U-Net. U-Net says: “compress, then recover resolution with a decoder.” DeepLabV3 says: “keep a strong semantic backbone, enlarge the receptive field intelligently, and aggregate context across several scales before producing the dense mask.”

That difference is important for students. U-Net is often easier to understand at first glance, but DeepLabV3 is closer to how many modern segmentation systems are built on top of powerful pretrained backbones.

### 6.1 Why plain CNN downsampling is not enough

If we keep stacking pooling or strided convolution layers, we gain semantic abstraction, but lose spatial precision. For image classification that tradeoff is acceptable. For segmentation it becomes dangerous:

- small objects may disappear
- boundaries get washed out
- thin structures are lost
- the model becomes confident about the class but vague about the outline

DeepLabV3 tries to keep semantic strength without throwing away too much spatial information.

### 6.2 Atrous / dilated convolution

The first key idea is the **dilated convolution** (also called **atrous convolution**).

In an ordinary convolution, the kernel samples neighboring pixels densely. In a dilated convolution, the sampling points are spaced out. That means the filter covers a larger effective field of view without dramatically increasing the number of weights.

Students often ask why this is useful. The short answer is:

- the model can “look farther” into the scene
- but it still keeps a feature map resolution that is useful for segmentation

This matters because segmentation decisions are often ambiguous locally. A single fur pixel, road pixel, or plant pixel may look uncertain in isolation. Once the model sees a wider neighborhood, the label becomes easier to infer.

### 6.3 Receptive field and effective context

In segmentation, the concept of **receptive field** is extremely important. A pixel prediction should ideally use:

- local detail to place boundaries accurately
- large context to understand the object and scene

DeepLabV3 increases the effective receptive field by combining dilated convolutions with multi-scale context aggregation. This is the heart of the architecture.

### 6.4 Output stride

Another term students meet in DeepLab papers is **output stride**. This means how much the spatial resolution was reduced relative to the input.

For example:

- output stride 16 means the feature map is 16 times smaller in each spatial dimension than the input
- output stride 8 keeps more detail but costs more memory and compute

Why should students care?

Because output stride is one of the practical levers that trade:

- speed and memory
- against
- boundary quality and small-object sensitivity

### 6.5 ASPP: Atrous Spatial Pyramid Pooling

The signature DeepLabV3 block is **ASPP**.

Instead of trusting one single receptive field, ASPP builds several parallel branches:

- one branch may use a small or zero dilation
- another branch may use a medium dilation
- another branch may use a larger dilation
- a global pooling branch may summarize image-level context

These branches are then fused together.

The intuition is simple but powerful: real images contain patterns at many scales. A cat face, a pet body, the outline against the floor, and the global indoor scene all live at different spatial scales. ASPP gives the model a structured way to capture all of them.

### 6.6 Why DeepLabV3 often feels strong in practice

DeepLabV3 is attractive because it combines:

- a strong pretrained backbone
- large context through dilated convolutions
- explicit multi-scale aggregation through ASPP

This often gives a very strong baseline even before heavy architecture customization.

For students, it is helpful to compare the styles:

- U-Net emphasizes recovering detail through skip connections
- DeepLabV3 emphasizes preserving semantic quality while widening context

Both are valid ways to address the same dense prediction tension.

### 6.7 DeepLabV3 vs U-Net: when each feels natural

U-Net often feels natural when:

- the dataset is not huge
- boundary precision is central
- medical imaging or small custom datasets are involved
- we want an architecture that is easy to explain and customize

DeepLabV3 often feels natural when:

- we want to leverage a strong pretrained classification backbone
- scene context matters a lot
- we want a strong off-the-shelf baseline from `torchvision`
- we expect multi-scale context to help significantly

This is not a rigid rule, but it is a useful teaching heuristic.

### 6.8 What students should remember architecturally

If someone forgets all implementation details and only remembers one sentence, let it be this:

**DeepLabV3 is a segmentation model that keeps strong convolutional features, expands the effective receptive field with atrous convolutions, and fuses several context scales through ASPP.**

That single sentence captures the architecture better than memorizing a long list of block names.

### 6.9 Typical strengths of DeepLabV3

- strong semantic understanding
- good leverage of pretrained backbones
- strong performance on many benchmark-style datasets
- elegant multi-scale context handling

### 6.10 Typical weaknesses or tradeoffs

- not always as intuitive as U-Net for students at first glance
- can still produce coarse boundaries if the decoder/head is too simple
- output stride and memory tradeoffs matter
- for tiny datasets, a simpler custom architecture may sometimes be easier to control

### 6.11 Why it still fits a beginner-friendly notebook

Even though DeepLabV3 is more modern and slightly more abstract than U-Net, it belongs in a beginner-friendly notebook because it teaches several foundational ideas:

- receptive field
- multi-scale features
- dense prediction with pretrained backbones
- how segmentation models differ from classification CNNs

That makes it a very valuable “next step” architecture once students understand the encoder-decoder idea.


## 7. Output tensors and losses

Students often ask: what does the model actually produce?

For semantic segmentation with `C` classes, the raw output is usually a tensor of shape:

`(batch, C, height, width)`

That means:

- for every image in the batch  
- for every class  
- for every pixel location  
- the model produces a score

These are usually called **logits**. The class prediction for each pixel is the class with the highest logit.

### Cross-entropy loss

The most common loss is pixel-wise cross-entropy. It compares the predicted class distribution at each pixel with the ground-truth class index.

### Dice-style losses

Dice is common when the target object is small or class imbalance is severe, for example in medical imaging. It directly rewards overlap quality.

### Why this matters in practice

If the foreground is tiny and the background dominates the image, naive training can make the model biased toward background. This is why segmentation practitioners often monitor IoU or Dice rather than only raw accuracy.


In [ ]:
if not TORCH_AVAILABLE:
    print("PyTorch is required for the segmentation pipeline.")
else:
    from torchvision.datasets import OxfordIIITPet
    from torchvision.transforms import v2 as T

    DATA_ROOT = Path.cwd() / "data" / "oxford_pet"
    IMAGE_SIZE = 160
    BATCH_SIZE = 8
    NUM_CLASSES = 3  # background, pet, outline

    image_tf = T.Compose(
        [
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToImage(),
            T.ToDtype(torch.float32, scale=True),
        ]
    )

    mask_tf = T.Compose(
        [
            T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.NEAREST),
            T.ToImage(),
        ]
    )

    class OxfordPetSegmentation(Dataset):
        def __init__(self, root: Path, split: str = "trainval"):
            self.ds = OxfordIIITPet(
                root=root,
                split=split,
                target_types="segmentation",
                download=True,
            )

        def __len__(self):
            return len(self.ds)

        def __getitem__(self, idx):
            image, trimap = self.ds[idx]
            image = image_tf(image)
            trimap = mask_tf(trimap).squeeze(0).long()
            trimap = torch.clamp(trimap - 1, min=0, max=2)
            return image, trimap

    pet_train = OxfordPetSegmentation(DATA_ROOT, split="trainval")
    pet_test = OxfordPetSegmentation(DATA_ROOT, split="test")

    print("Train samples:", len(pet_train))
    print("Test samples :", len(pet_test))


In [ ]:
if not TORCH_AVAILABLE:
    print("Skipping visualization: PyTorch not available.")
else:
    sample_image, sample_mask = pet_train[0]
    print("Image tensor shape:", tuple(sample_image.shape))
    print("Mask tensor shape :", tuple(sample_mask.shape))
    print("Mask labels       :", torch.unique(sample_mask).tolist())

    if MPL_AVAILABLE:
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))
        axes[0].imshow(sample_image.permute(1, 2, 0))
        axes[0].set_title("Oxford-IIIT Pet image")
        axes[0].axis("off")

        axes[1].imshow(sample_mask, cmap="viridis")
        axes[1].set_title("Trimap / class mask")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()


## 8. Metrics students must not ignore

### Pixel accuracy

The fraction of correctly predicted pixels. Easy to compute, but often too optimistic.

### IoU

$$
IoU = \frac{TP}{TP + FP + FN}
$$

IoU asks how well predicted and true regions overlap. It penalizes false positives and false negatives in a more balanced way than plain accuracy.

### Dice

$$
Dice = \frac{2TP}{2TP + FP + FN}
$$

Dice is similar in spirit to IoU and is especially popular in domains where the positive region is small.

If students remember one lesson here, it should be this:

**high pixel accuracy does not automatically mean good masks.**


In [ ]:
if not TORCH_AVAILABLE:
    print("Skipping model creation: PyTorch not available.")
else:
    from torchvision.models.segmentation import (
        DeepLabV3_ResNet50_Weights,
        deeplabv3_resnet50,
    )

    weights = DeepLabV3_ResNet50_Weights.DEFAULT
    model = deeplabv3_resnet50(weights=weights)
    model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total params    :", f"{total_params:,}")
    print("Trainable params:", f"{trainable_params:,}")


## 9. Real dataset example: Oxford-IIIT Pet

The lecture uses Oxford-IIIT Pet because it is a strong teaching dataset:

- real photographs rather than toy geometry  
- segmentation trimaps rather than only labels  
- visually easy to interpret  
- small enough for classroom-scale experiments

The trimap format is also educational because it highlights a subtle point: datasets do not always hand us masks in the exact format our model expects. A real pipeline often needs small preprocessing steps to standardize labels.


In [ ]:
if not TORCH_AVAILABLE:
    print("Skipping training loop: PyTorch not available.")
else:
    SMALL_TRAIN_SIZE = 120
    SMALL_VAL_SIZE = 32
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    small_train = torch.utils.data.Subset(pet_train, list(range(SMALL_TRAIN_SIZE)))
    small_val = torch.utils.data.Subset(pet_test, list(range(SMALL_VAL_SIZE)))

    train_loader = DataLoader(small_train, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(small_val, batch_size=BATCH_SIZE, shuffle=False)

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    def evaluate_pixel_accuracy(loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, masks in loader:
                images = images.to(DEVICE)
                masks = masks.to(DEVICE)
                logits = model(images)["out"]
                preds = logits.argmax(dim=1)
                correct += (preds == masks).sum().item()
                total += masks.numel()
        return correct / max(total, 1)

    model.train()
    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)["out"]
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        break

    val_acc = evaluate_pixel_accuracy(val_loader)
    print("One-step demo complete.")
    print("Validation pixel accuracy after the demo step:", round(val_acc, 4))


## 10. Why we use DeepLabV3 in the code

The notebook chooses DeepLabV3-ResNet50 from `torchvision` for the practical part, not because it is the only segmentation architecture worth teaching, but because it is a good classroom compromise:

- pretrained weights exist  
- the API is stable  
- the architecture is recognizable in the literature  
- students can swap the final classifier and fine-tune quickly

This lets the notebook focus on learning the segmentation workflow instead of spending all the time on low-level model plumbing.


## 11. Applications and real deployment thinking

Segmentation is useful whenever the shape of the object matters, not just its presence.

Typical applications:

- autonomous driving: road, curb, lane, pedestrian area  
- medicine: tumors, organs, lesions, vessels  
- satellite imagery: roads, buildings, water, vegetation  
- robotics: free space and manipulable objects  
- agriculture: plants, weeds, damaged leaf regions

A useful engineering rule is:

- if a box is enough, detection is cheaper  
- if boundaries matter, segmentation is the right tool


In [ ]:
if not TORCH_AVAILABLE:
    print("Skipping inference demo: PyTorch not available.")
else:
    model.eval()
    demo_image, demo_mask = pet_test[1]

    with torch.no_grad():
        logits = model(demo_image.unsqueeze(0).to(DEVICE))["out"]
        pred_mask = logits.argmax(dim=1).squeeze(0).cpu()

    if MPL_AVAILABLE:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(demo_image.permute(1, 2, 0))
        axes[0].set_title("Input image")
        axes[0].axis("off")

        axes[1].imshow(demo_mask, cmap="viridis")
        axes[1].set_title("Ground truth mask")
        axes[1].axis("off")

        axes[2].imshow(pred_mask, cmap="viridis")
        axes[2].set_title("Predicted mask")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()


## 12. Common mistakes and failure modes

Students often make the same errors the first time they train a segmentation model:

- resizing masks with the wrong interpolation  
- trusting accuracy too much  
- forgetting to visualize predictions  
- ignoring class imbalance  
- using too few examples and expecting sharp masks

Typical qualitative failures include:

- coarse boundaries  
- small objects disappearing  
- holes inside objects  
- correct object class with incorrect shape

This is why side-by-side visualization of image, mask, and prediction is not optional. It is a core debugging step.


In [ ]:
# Exercise scaffold: switch from DeepLabV3 to a custom U-Net

if not TORCH_AVAILABLE:
    print("PyTorch is required for the U-Net exercise.")
else:
    from importlib.util import module_from_spec, spec_from_file_location

    candidate_paths = [
        Path.cwd().parent / "lecture13. PyTorch" / "U-Net" / "unet_network.py",
        Path("/Users/hiber/repositories/courses/ml/ml_course_en/lecture13. PyTorch/U-Net/unet_network.py"),
    ]

    unet_path = next((p for p in candidate_paths if p.exists()), None)
    if unet_path is None:
        print("Could not find local U-Net implementation.")
    else:
        spec = spec_from_file_location("local_unet", unet_path)
        module = module_from_spec(spec)
        spec.loader.exec_module(module)
        unet = module.UNet(in_channels=3, out_channels=NUM_CLASSES, init_features=16)
        print(unet.__class__.__name__)


## Exercises

1. Replace DeepLabV3 with U-Net and compare the speed/quality tradeoff.  
2. Compute IoU per class instead of only pixel accuracy.  
3. Increase image size and observe the effect on memory and mask sharpness.  
4. Train longer and inspect which improvements appear first: better global shape or better edges.


## References

- Expanded from the course materials and the Russian notes at [deepmachinelearning.ru](https://deepmachinelearning.ru/docs/Neural-networks/Object-detection).  
- `torchvision` documentation for Oxford-IIIT Pet and segmentation models.  
- U-Net paper: *U-Net: Convolutional Networks for Biomedical Image Segmentation*.  
- DeepLab line of work for dilated convolutions and ASPP.
